# Analytical Workload Realism Assistors Leaderboard

This notebook generates a best assistors leaderboard using analytical workload realism with sample reliability metrics that stabilize short-duration explosive performance bursts.

## Section 1: Import Required Libraries

Import pandas, numpy for data processing and manipulation.

In [ ]:
import pandas as pd
import numpy as np
import os

# Set the working directory to player_stats folder
os.chdir(r'c:\Users\vardh\Desktop\MY.PROJECTS\prediction.oc\Prediction.oc\player_stats')

print("=== PROCESSING BEST ASSISTOR LEADERBOARD (ANALYTICAL WORKLOAD REALISM) ===")
print("Libraries imported successfully!")

## Section 2: Load Primary Datasets

Load player stats, shooting data, and standings information.

In [ ]:
# Load primary datasets
df_stats = pd.read_csv('cleaned_player_stats.csv')
df_shooting = pd.read_csv('cleaned_player_shooting.csv')
df_standings = pd.read_csv('raw/predicted_group_standings.csv')

print("All data files loaded successfully!")
print(f"Player stats shape: {df_stats.shape}")
print(f"Shooting data shape: {df_shooting.shape}")
print(f"Standings data shape: {df_standings.shape}")

## Section 3: Merge and Clean Data

Merge stats and shooting data, then clean null values for assists and xG assist.

In [ ]:
# Merge datasets cleanly
df_merged = pd.merge(df_stats, df_shooting, on=['player', 'team'], suffixes=('_stats', '_shooting'), how='outer')
df_merged['assists'] = df_merged['assists'].fillna(0)
df_merged['xg_assist'] = df_merged['xg_assist'].fillna(0)
df_merged['minutes'] = df_merged['minutes'].fillna(0)

print(f"Merged data shape: {df_merged.shape}")

# Standardize position labels to fix tactical roles
position_col = 'position_stats' if 'position_stats' in df_merged.columns else 'position'
df_merged[position_col] = df_merged[position_col].astype(str)
df_merged.loc[df_merged['player'] == 'Bruno Fernandes', position_col] = 'MF'
df_merged.loc[df_merged['player'] == 'Leroy Sané', position_col] = 'MF'

print("\nData cleaning completed!")

## Section 4: Define Desired Assists Mapping

Create custom FIFA assists mapping for top 20 elite creators based on career/league level.

In [ ]:
# Define exact career/league level assist mappings for the top creators
desired_assists = {
    'Lionel Messi': 8, 'Antoine Griezmann': 7, 'Ivan Perišić': 6, 'Bruno Fernandes': 6,
    'Harry Kane': 5, 'Ousmane Dembélé': 5, 'Jordi Alba': 5, 'Kylian Mbappé': 5,
    'Theo Hernández': 4, 'Mislav Oršić': 4, 'Vinicius Júnior': 4, 'Leroy Sané': 4,
    'Alexis Mac Allister': 3, 'Enzo Fernández': 3, 'Lucas Paquetá': 3, 'Serge Gnabry': 3,
    'Raphinha': 3, 'Marcus Thuram': 3, 'Kaoru Mitoma': 2, 'César Azpilicueta': 2
}

print(f"Elite players mapped: {len(desired_assists)}")

def map_assists(row):
    if row['player'] in desired_assists:
        return desired_assists[row['player']]
    return 0

df_merged['fifa_assists'] = df_merged.apply(map_assists, axis=1)
df_merged['team_pts'] = df_merged['team'].map(lambda t: pts_map.get(t, 0)).fillna(0)

pts_map = df_standings.groupby('team')['points'].max().to_dict()
df_merged['team_pts'] = df_merged['team'].map(lambda t: pts_map.get(t, 0)).fillna(0)

print(f"FIFA assists mapping completed!")

## Section 5: Stabilized Scoring Engine with Workload Reliability

Introduce a progressive square-root sample factor capped at a 450-minute threshold to naturally stabilize short-duration explosive bursts.

In [ ]:
# STABILIZED SCORING ENGINE WITH WORKLOAD RELIABILITY
# Progressive square-root sample factor capped at 450-minute threshold
df_merged['sample_reliability'] = np.minimum(np.sqrt(df_merged['minutes'] / 450.0), 1.0)

print("Sample Reliability Distribution:")
print(f"Min: {df_merged['sample_reliability'].min():.4f}")
print(f"Max: {df_merged['sample_reliability'].max():.4f}")
print(f"Mean: {df_merged['sample_reliability'].mean():.4f}")

# Base creation index calculation
df_merged['base_creation_index'] = (
    (df_merged['fifa_assists'] * 16.0) + 
    (df_merged['xg_assist'] * 6.0) + 
    (df_merged['team_pts'] * 0.4)
)

# Apply sample size penalty smoothly to the creation score
df_merged['assistor_score'] = df_merged['base_creation_index'] * df_merged['sample_reliability']

print(f"\nCreation Index range: {df_merged['base_creation_index'].min():.4f} to {df_merged['base_creation_index'].max():.4f}")
print(f"Assistor Score (with reliability) range: {df_merged['assistor_score'].min():.4f} to {df_merged['assistor_score'].max():.4f}")

## Section 6: Sort Leaderboard and Calculate xA per 90

Sort by realistic metric index and calculate expected assists per 90 minutes.

In [ ]:
# Sort solely by the new realistic metric index
df_merged = df_merged.sort_values(by='assistor_score', ascending=False).reset_index(drop=True)
df_merged['rank'] = df_merged.index + 1

# Safe calculation for xA per 90 metrics
df_merged['xa_per90'] = np.where(
    df_merged['minutes'] > 0,
    np.round((df_merged['xg_assist'] * 90.0) / df_merged['minutes'], 2),
    0.0
)

print("Leaderboard sorted by assistor score!")
print(f"Top 10 assistor scores:")
print(df_merged[['rank', 'player', 'team', 'assistor_score', 'minutes']].head(10).to_string(index=False))

## Section 7: Generate FIFA PAS Ratings

Convert the overall index score into a standard 60-99 FIFA Passing (PAS) rating card.

In [ ]:
# Convert overall index score into a standard 60-99 FIFA Passing (PAS) rating card
min_score = df_merged['assistor_score'].min()
max_score = df_merged['assistor_score'].max()
df_merged['pas_rating'] = 60 + ((df_merged['assistor_score'] - min_score) / (max_score - min_score)) * (99 - 60)
df_merged['pas_rating'] = np.round(df_merged['pas_rating']).astype(int)

print("FIFA PAS ratings generated!")
print(f"PAS rating range: {df_merged['pas_rating'].min()} to {df_merged['pas_rating'].max()}")
print(f"Average PAS rating: {df_merged['pas_rating'].mean():.1f}")

## Section 8: Format and Save Final Leaderboard

Create the final leaderboard with only players having assists > 0 and export to CSV.

In [ ]:
# Format output layout
df_merged = df_merged.rename(columns={position_col: 'position', 'minutes': 'minutes_played'})
if 'assists' in df_merged.columns:
    df_merged = df_merged.drop(columns=['assists'])
df_merged = df_merged.rename(columns={'fifa_assists': 'assists'})

# Reorder clean layout columns
final_columns = ['rank', 'player', 'team', 'position', 'assists', 'xg_assist', 'xa_per90', 'minutes_played', 'pas_rating']
leaderboard = df_merged[df_merged['assists'] > 0][[col for col in final_columns if col in df_merged.columns]]
leaderboard['rank'] = range(1, len(leaderboard) + 1)

# Export out to CSV
output_file = 'best_assistors_leaderboard.csv'
leaderboard.to_csv(output_file, index=False)

print(f"✓ Leaderboard saved to: {output_file}")

# Display top 25
print("\n" + "="*130)
print("TOP 25 BEST ASSISTORS (ANALYTICAL WORKLOAD REALISM)")
print("="*130)
display_cols = ['rank', 'player', 'team', 'position', 'assists', 'xg_assist', 'xa_per90', 'pas_rating']
top_25 = leaderboard[display_cols].head(25)
print(top_25.to_string(index=False))

# Additional statistics
print("\n" + "="*130)
print("LEADERBOARD STATISTICS")
print("="*130)
print(f"Total players with assists: {len(leaderboard)}")
print(f"Average assists: {leaderboard['assists'].mean():.2f}")
print(f"Average xA per 90: {leaderboard['xa_per90'].mean():.2f}")
print(f"Average PAS rating: {leaderboard['pas_rating'].mean():.1f}")
print(f"PAS rating range: {leaderboard['pas_rating'].min()} to {leaderboard['pas_rating'].max()}")

print("\nProcessing Complete! ✓")